In [ ]:
import sys
sys.path.append("..")
import pandas as pd
import plotly.express as px
from pipeline.fetchers.keepa_fetcher import KeepaFetcher
from pipeline.fetchers.google_trends import GoogleTrendsFetcher
from pipeline.fetchers.reddit_fetcher import RedditFetcher
from pipeline.processors.trend_score import TrendScoreCalculator
from pipeline.outputs.export import DataExporter
from pipeline.config import PET_SUPPLIES_SEARCH_TERMS
print("Setup complete")

In [ ]:
gt = GoogleTrendsFetcher()
trend_results = []
for term in PET_SUPPLIES_SEARCH_TERMS[:10]:
    result = gt.get_trend_direction(term)
    trend_results.append(result)
    
trend_df = pd.DataFrame(trend_results).sort_values("change_pct", ascending=False)
trend_df

In [ ]:
fig = px.bar(trend_df, x="keyword", y="change_pct", color="direction",
             title="Pet Supplies Search Trend Direction (3-Month)",
             color_discrete_map={"rising": "green", "falling": "red", "stable": "gray", "insufficient_data": "lightgray"})
fig.show()

In [ ]:
rf = RedditFetcher()
reddit_results = []
for term in ["grain free dog food", "automatic cat feeder", "dog enrichment toys", "cat water fountain", "pet camera monitor"]:
    signal = rf.get_signal_strength(term)
    reddit_results.append(signal)

pd.DataFrame(reddit_results)

In [ ]:
calc = TrendScoreCalculator()
results = []

for term in PET_SUPPLIES_SEARCH_TERMS[:8]:
    trends_df = gt.fetch_interest_over_time([term], timeframe="today 3-m")
    reddit_signal = rf.get_signal_strength(term)
    
    score = calc.calculate(
        keyword=term,
        bsr_df=pd.DataFrame(),
        trends_df=trends_df,
        reddit_signal=reddit_signal,
    )
    results.append(score)

scores_df = pd.DataFrame(results)[["keyword", "trend_score", "data_quality"]].sort_values("trend_score", ascending=False)
scores_df

In [ ]:
exporter = DataExporter()
exporter.export_trend_scores(results, "pet_supplies_v1_trend_scores.json")
print(f"Exported {len(results)} trend scores to data/exports/")

# Pet Supplies Trend Analysis — Key Findings

## Highest Signal Categories
The TrendScore analysis reveals which pet sub-niches are showing the strongest cross-platform momentum.

## Next Steps
- Cross-reference TrendScores with actual BSR data (needs Keepa API access)
- Deep-dive into top 3 categories with review gap analysis
- Set up weekly automated trend refresh